# NB02 — EDA: Restaurants

**Phase 2, Notebook 1 of 5.** Forensic exploration of the 717 Italian restaurants across Philadelphia and Tampa. Produces four figures saved to `outputs/figures/nb02/` and a summary card for the horizon scan's data section.

| Figure | Description |
|---|---|
| `fig1_stars_by_city.png` | Star rating distribution by city |
| `fig2_review_count_dist.png` | Review count distribution (log scale) |
| `fig3_price_tier_by_city.png` | Price tier breakdown by city |
| `fig4_open_closed.png` | Open vs closed ratio by city |

Run cells top-to-bottom. Inspect each figure before proceeding to the next.

## 1. Environment setup

Imports, project root resolution, and output directory creation.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np

ROOT      = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

PROCESSED = ROOT / "data" / "processed"
FIG_DIR   = ROOT / "outputs" / "figures" / "nb02"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Consistent style for all figures
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

print(f"Project root : {ROOT}")
print(f"Figures dir  : {FIG_DIR}")

## 2. Load restaurants.parquet and inspect

Reads the filtered restaurant dataset and prints a concise summary. Check the price_range column — it contains string digits ('1'–'4') plus nulls that need cleaning before plotting.

In [ ]:
df = pd.read_parquet(PROCESSED / "restaurants.parquet")
print(f"Shape  : {df.shape}")
print(f"Cities : {df['city'].value_counts().to_dict()}")
print(f"Stars  : min={df['stars'].min()}  max={df['stars'].max()}  mean={df['stars'].mean():.2f}")
print(f"Reviews: median={df['review_count'].median():.0f}  max={df['review_count'].max()}")
print()

# Clean price_range: coerce to int, drop unparseable
df["price"] = pd.to_numeric(df["price_range"], errors="coerce")
print("Price tier distribution (after cleaning):")
print(df["price"].value_counts(dropna=False).sort_index().to_string())
print()
print(df.head(3).to_string())

## 3. Figure 1 — Star rating distribution by city

Grouped bar chart: for each star value (1.0–5.0), how many restaurants in Philadelphia vs Tampa? Shows whether the two metros have similar quality distributions or are skewed differently.

In [ ]:
star_counts = (
    df.groupby(["city", "stars"])
    .size()
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=star_counts, x="stars", y="count", hue="city", ax=ax)
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of restaurants")
ax.set_title("Star rating distribution — Philadelphia vs Tampa")
ax.legend(title="City")

out = FIG_DIR / "fig1_stars_by_city.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")

## 4. Figure 2 — Review count distribution (log scale)

Histogram of review counts per restaurant on a log x-axis. Review count is a proxy for restaurant popularity and Yelp visibility. Expect a heavy right skew — a few very-reviewed restaurants and many with few reviews.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for ax, (city, grp) in zip(axes, df.groupby("city")):
    ax.hist(grp["review_count"], bins=40, edgecolor="white", linewidth=0.4)
    ax.set_xscale("log")
    ax.set_xlabel("Review count (log scale)")
    ax.set_ylabel("Number of restaurants")
    ax.set_title(city)
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    med = grp["review_count"].median()
    ax.axvline(med, color="red", linestyle="--", linewidth=1.2, label=f"Median = {med:.0f}")
    ax.legend(fontsize=8)

fig.suptitle("Review count distribution by city", y=1.01)
fig.tight_layout()

out = FIG_DIR / "fig2_review_count_dist.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")

## 5. Figure 3 — Price tier breakdown by city

Stacked bar chart showing the proportion of restaurants at each Yelp price tier ($ to $$$$) for each city. Price tier is the primary segmentation signal the market analyst agent uses when building competitor profiles.

In [ ]:
# Rows with a valid price tier only
priced = df.dropna(subset=["price"]).copy()
priced["price"] = priced["price"].astype(int)

# Use escaped dollar signs so matplotlib's mathtext parser does not choke
price_labels = {1: r"\$", 2: r"\$\$", 3: r"\$\$\$", 4: r"\$\$\$\$"}
priced["tier"] = priced["price"].map(price_labels)

tier_order = [r"\$", r"\$\$", r"\$\$\$", r"\$\$\$\$"]
pivot = (
    priced.groupby(["city", "tier"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=tier_order, fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(7, 4))
pivot_pct.plot(kind="bar", stacked=True, ax=ax, colormap="Blues",
               edgecolor="white", linewidth=0.5)
ax.set_xlabel("City")
ax.set_ylabel("Percentage of restaurants")
ax.set_title("Price tier distribution by city")
ax.set_xticklabels(pivot_pct.index, rotation=0)
ax.legend(title="Tier", bbox_to_anchor=(1.01, 1), loc="upper left")

# Annotate with counts
for i, city in enumerate(pivot_pct.index):
    total = pivot.loc[city].sum()
    ax.text(i, 103, f"n={total}", ha="center", fontsize=8)

out = FIG_DIR / "fig3_price_tier_by_city.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
print("Count table:")
print(pivot.to_string())
print()
print(f"Restaurants excluded (no price tier): {df['price'].isna().sum()}")

## 6. Figure 4 — Open vs closed ratio by city

The Yelp dataset covers years of data, so many listed restaurants are now permanently closed. This ratio matters for the scan: the agent corpus includes closed restaurants, which may produce outdated or irrelevant competitor signals. Documented as a data-honesty limitation.

In [ ]:
status_map = {1: "Open", 0: "Closed"}
df["status"] = df["is_open"].map(status_map)

status_counts = (
    df.groupby(["city", "status"])
    .size()
    .reset_index(name="count")
)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
colors = {"Open": "#4c9be8", "Closed": "#d9534f"}

for ax, (city, grp) in zip(axes, status_counts.groupby("city")):
    grp = grp.set_index("status").reindex(["Open", "Closed"], fill_value=0)
    bars = ax.bar(grp.index, grp["count"],
                  color=[colors[s] for s in grp.index],
                  edgecolor="white")
    for bar, val in zip(bars, grp["count"]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 2, str(val),
                ha="center", va="bottom", fontsize=9)
    total = grp["count"].sum()
    pct_open = grp.loc["Open", "count"] / total * 100
    ax.set_title(f"{city}\n({pct_open:.0f}% open)")
    ax.set_ylabel("Restaurants")
    ax.set_ylim(0, grp["count"].max() * 1.15)

fig.suptitle("Open vs closed restaurants by city", y=1.01)
fig.tight_layout()

out = FIG_DIR / "fig4_open_closed.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")

## 7. Summary card — NB02

Concise dataset card for the horizon scan's data section. Also documents honest limitations of this corpus.

In [ ]:
print("=" * 58)
print("NB02 SUMMARY — Restaurant corpus")
print("=" * 58)

for city, grp in df.groupby("city"):
    priced_grp = grp.dropna(subset=["price"])
    print(f"\n── {city} ({len(grp)} restaurants) ──")
    print(f"  Stars        mean={grp['stars'].mean():.2f}  "
          f"median={grp['stars'].median():.1f}")
    print(f"  Review count median={grp['review_count'].median():.0f}  "
          f"max={grp['review_count'].max()}")
    tier_dist = priced_grp["price"].astype(int).value_counts().sort_index().to_dict()
    print(f"  Price tiers  {tier_dist}  "
          f"({grp['price'].isna().sum()} missing)")
    open_pct = grp["is_open"].mean() * 100
    print(f"  Open         {open_pct:.0f}%")

print()
print("── Data honesty ──")
n_missing_price = df["price"].isna().sum()
n_closed = (df["is_open"] == 0).sum()
print(f"  {n_missing_price} restaurants ({n_missing_price/len(df)*100:.1f}%) have no price tier.")
print(f"  {n_closed} restaurants ({n_closed/len(df)*100:.1f}%) are permanently closed.")
print("  Closed restaurants are retained in the corpus but flagged.")
print("  Yelp data snapshot: no exact collection date, estimated 2019–2022.")
print("  Coverage: Yelp-listed restaurants only — no dark kitchens, pop-ups,")
print("  or restaurants that never created a Yelp profile.")
print()
print("── Figures saved ──")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  {f.name}")
print()
print("Next: NB02b_eda_reviews.ipynb")